In [1]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface
import spikeinterface.exporters as sexp
from spikeinterface.core import write_binary_recording
from pathlib import Path
import pickle
from utils_clique import (
    CliqueInfo,
    build_shank_cliques,
    neuron_inf_dict_to_dataframe,
    get_recording_clique,
    filter_neuron_inf_by_clique,
    filter_gt_detect_array_by_clique,
    prepare_training_data,
    train_autosort_model
)


In [2]:
file_dict = {
    'mouse1': {
        1215: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse1_natima_251215_232216',
        1217: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse1_natima_251217_225054',
        1219: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse1&oldV1_natima_251219_201746'
    },
    'mouse2': {
        1214: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse&V1B_natima_251214_154409',
        1215: '/media/ubuntu/sda/mouse_test/raw_data/WLF_V1left&128ch2mouse_natima_251215_223556',
        1216: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse&V1left_natima_251216_214224',
        1217: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse2&V1left_natima_251217_220244',
        1218: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse2&V1od_natima_251218_214009',
        1219: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse2&V1left_natima_251219_192148'
    }

}

In [3]:
channel_list_A = ['A-000', 'A-001', 'A-002',
       'A-003', 'A-004', 'A-005', 'A-006', 'A-007', 'A-008', 'A-009',
       'A-010', 'A-011', 'A-012', 'A-013', 'A-014', 'A-015', 'A-016',
       'A-017', 'A-018', 'A-019', 'A-020', 'A-021', 'A-022', 'A-023',
       'A-024', 'A-025', 'A-026', 'A-027', 'A-028', 'A-029', 'A-030',
       'A-031', 'A-032', 'A-033', 'A-034', 'A-035', 'A-036', 'A-037',
       'A-038', 'A-039', 'A-040', 'A-041', 'A-042', 'A-043', 'A-044',
       'A-045', 'A-046', 'A-047', 'A-048', 'A-049', 'A-050', 'A-051',
       'A-052', 'A-053', 'A-054', 'A-055', 'A-056', 'A-057', 'A-058',
       'A-059', 'A-060', 'A-061', 'A-062', 'A-063', 'A-064', 'A-065',
       'A-066', 'A-067', 'A-068', 'A-069', 'A-070', 'A-071', 'A-072',
       'A-073', 'A-074', 'A-075', 'A-076', 'A-077', 'A-078', 'A-079',
       'A-080', 'A-081', 'A-082', 'A-083', 'A-084', 'A-085', 'A-086',
       'A-087', 'A-088', 'A-089', 'A-090', 'A-091', 'A-092', 'A-093',
       'A-094', 'A-095', 'A-096', 'A-097', 'A-098', 'A-099', 'A-100',
       'A-101', 'A-102', 'A-103', 'A-104', 'A-105', 'A-106', 'A-107',
       'A-108', 'A-109', 'A-110', 'A-111', 'A-112', 'A-113', 'A-114',
       'A-115', 'A-116', 'A-117', 'A-118', 'A-119', 'A-120', 'A-121',
       'A-122', 'A-123', 'A-124', 'A-125', 'A-126', 'A-127']

channel_list_B = ['B-000', 'B-001', 'B-002',
       'B-003', 'B-004', 'B-005', 'B-006', 'B-007', 'B-008', 'B-009',
       'B-010', 'B-011', 'B-012', 'B-013', 'B-014', 'B-015', 'B-016',
       'B-017', 'B-018', 'B-019', 'B-020', 'B-021', 'B-022', 'B-023',
       'B-024', 'B-025', 'B-026', 'B-027', 'B-028', 'B-029', 'B-030',
       'B-031', 'B-032', 'B-033', 'B-034', 'B-035', 'B-036', 'B-037',
       'B-038', 'B-039', 'B-040', 'B-041', 'B-042', 'B-043', 'B-044',
       'B-045', 'B-046', 'B-047', 'B-048', 'B-049', 'B-050', 'B-051',
       'B-052', 'B-053', 'B-054', 'B-055', 'B-056', 'B-057', 'B-058',
       'B-059', 'B-060', 'B-061', 'B-062', 'B-063', 'B-064', 'B-065',
       'B-066', 'B-067', 'B-068', 'B-069', 'B-070', 'B-071', 'B-072',
       'B-073', 'B-074', 'B-075', 'B-076', 'B-077', 'B-078', 'B-079',
       'B-080', 'B-081', 'B-082', 'B-083', 'B-084', 'B-085', 'B-086',
       'B-087', 'B-088', 'B-089', 'B-090', 'B-091', 'B-092', 'B-093',
       'B-094', 'B-095', 'B-096', 'B-097', 'B-098', 'B-099', 'B-100',
       'B-101', 'B-102', 'B-103', 'B-104', 'B-105', 'B-106', 'B-107',
       'B-108', 'B-109', 'B-110', 'B-111', 'B-112', 'B-113', 'B-114',
       'B-115', 'B-116', 'B-117', 'B-118', 'B-119', 'B-120', 'B-121',
       'B-122', 'B-123', 'B-124', 'B-125', 'B-126', 'B-127']

In [4]:
mouse_name = 'mouse1'
date = 1215
data_path = '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse1_natima_251215_232216'
output_dir = '/media/ubuntu/sda/mouse_test/sorted/mountainsort/mouse1/1215'

In [5]:
print(f"\n{'='*60}")
print(f"处理: {mouse_name}, 日期: {date}, 路径: {data_path}")
print(f"{'='*60}")

# 获取该数据路径下的所有rhd文件
file_list_path = Path(data_path)
rhd_files = list(file_list_path.glob("*.rhd"))
file_list = sorted(rhd_files)


# 读取并合并所有rhd文件
recording_raw_list = []
for file in file_list:
    recording_raw_list.append(se.read_intan(file, stream_id='0'))
recording_raw = concatenate_recordings(recording_list=recording_raw_list)

# 检测通道类型并选择对应的channel_list
available_channels = recording_raw.get_channel_ids()
if 'A-127' in available_channels:
    channel_list = channel_list_A
    print(f"检测到A通道，使用channel_list_A")
elif 'B-127' in available_channels:
    channel_list = channel_list_B
    print(f"检测到B通道，使用channel_list_B")


output_folder = f'/media/ubuntu/sda/mouse_test/sorted/mountainsort/{mouse_name}/{date}'

recording_raw = recording_raw.select_channels(channel_list)

recording_raw = spre.unsigned_to_signed(recording_raw)
recording_raw = spre.resample(recording_raw, 10000)
recording_recorded = spre.bandpass_filter(recording_raw, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_recorded, freq=50)
recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

probe = read_probeinterface('/media/ubuntu/sda/mouse_test/probe/tip_probe_128_1.json')
recording_f = recording_f.set_probegroup(probe)




处理: mouse1, 日期: 1215, 路径: /media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse1_natima_251215_232216
检测到B通道，使用channel_list_B


In [6]:
# import spikeinterface as si
# import numpy as np
# from spikeinterface.core import get_template_extremum_channel
# import scipy.spatial.distance
# from scipy.sparse.csgraph import connected_components
# import pickle
# import pandas as pd


# sorting_curated_phy = se.read_phy(f"{output_dir}/phy_folder_for_kilosort", exclude_cluster_groups=["noise"])
# analyzer_curated_phy = si.create_sorting_analyzer(
#     sorting=sorting_curated_phy, 
#     recording=recording_f, 
#     format='binary_folder',
#     folder=output_folder + '/analyzer_curated',
#     n_jobs=20
# )

# extensions_to_compute = [
#     "random_spikes",
#     "waveforms",
#     "noise_levels",
#     "templates",
#     "unit_locations",
#     "spike_locations",
#     "correlograms",
#     "template_similarity"
# ]

# extension_params = {
#     "random_spikes": {"method": "all"},
#     "unit_locations": {"method": "center_of_mass"},
#     "spike_locations": {"ms_before": 0.1},
#     "correlograms": {"bin_ms": 0.1},
#     "template_similarity": {"method": "cosine_similarity"}
# }

# analyzer_curated_phy.compute(extensions_to_compute, extension_params=extension_params, n_jobs=20)

# templates_ext = analyzer_curated_phy.get_extension("templates")
# templates_dense = templates_ext.data["average"]
# sparsity = analyzer_curated_phy.sparsity
# unit_locations_ext = analyzer_curated_phy.get_extension("unit_locations")
# unit_locations = unit_locations_ext.get_data()
# channel_locations = analyzer_curated_phy.get_channel_locations()

# if unit_locations.shape[1] >= 2:
#     unit_distances = scipy.spatial.distance.cdist(
#         unit_locations[:, :2], 
#         unit_locations[:, :2], 
#         metric="euclidean"
#     )
# else:
#     unit_distances = scipy.spatial.distance.cdist(
#         unit_locations, 
#         unit_locations, 
#         metric="euclidean"
#     )

# template_similarity_ext = analyzer_curated_phy.get_extension("template_similarity")
# template_similarity = template_similarity_ext.get_data()

# distance_threshold = 10.0
# similarity_threshold = 0.95
# num_units = len(analyzer_curated_phy.unit_ids)
# pair_mask = np.zeros((num_units, num_units), dtype=bool)

# for i in range(num_units):
#     for j in range(i + 1, num_units):
#         if unit_distances[i, j] < distance_threshold and template_similarity[i, j] > similarity_threshold:
#             pair_mask[i, j] = True
#             pair_mask[j, i] = True

# n_components, labels = connected_components(
#     csgraph=pair_mask, 
#     directed=False, 
#     return_labels=True
# )

# merge_unit_groups = []
# unit_ids_list = analyzer_curated_phy.unit_ids
# for component_id in range(n_components):
#     unit_indices = np.where(labels == component_id)[0]
#     if len(unit_indices) > 1:
#         group = [unit_ids_list[i] for i in unit_indices]
#         merge_unit_groups.append(group)

# if len(merge_unit_groups) > 0:
#     analyzer_merged = analyzer_curated_phy.merge_units(
#         merge_unit_groups=merge_unit_groups,
#         censor_ms=0.3,
#         merging_mode="hard",
#         new_id_strategy="append",
#         format='binary_folder',
#         folder=output_folder + '/analyzer_merged',
#         verbose=True,
#         n_jobs=20
#     )
    
#     analyzer_merged.compute(extensions_to_compute, extension_params=extension_params, n_jobs=20)
    
#     templates_ext_merged = analyzer_merged.get_extension("templates")
#     templates_dense_merged = templates_ext_merged.data["average"]
#     sparsity_merged = analyzer_merged.sparsity
#     unit_locations_ext_merged = analyzer_merged.get_extension("unit_locations")
#     unit_locations_merged = unit_locations_ext_merged.get_data()
#     channel_locations_merged = analyzer_merged.get_channel_locations()
    
#     position_waveforms_merged = []
#     unit_ids_list_merged = analyzer_merged.unit_ids
    
#     for unit_id in unit_ids_list_merged:
#         unit_index = analyzer_merged.sorting.id_to_index(unit_id)
#         template_dense_unit = templates_dense_merged[unit_index, :, :]
#         template_sparse_unit = sparsity_merged.sparsify_waveforms(template_dense_unit[np.newaxis, :, :], unit_id)[0]
#         sparse_channel_indices = sparsity_merged.unit_id_to_channel_indices[unit_id]
        
#         if len(sparse_channel_indices) == 0:
#             position_waveform = np.zeros(templates_dense_merged.shape[1], dtype=templates_dense_merged.dtype)
#             position_waveforms_merged.append(position_waveform)
#             continue
        
#         sparse_channel_locations = channel_locations_merged[sparse_channel_indices, :2]
#         unit_location = unit_locations_merged[unit_index, :2]
        
#         distances = np.sqrt(np.sum((sparse_channel_locations - unit_location[np.newaxis, :])**2, axis=1))
#         epsilon = 1e-10
#         weights = 1.0 / (distances + epsilon)
#         weights = weights / np.sum(weights)
        
#         position_waveform = np.dot(template_sparse_unit, weights)
#         position_waveforms_merged.append(position_waveform)
    
#     position_waveforms_merged = np.array(position_waveforms_merged)
    
#     extremum_channels_merged = get_template_extremum_channel(
#         analyzer_merged, 
#         peak_sign="neg",
#         outputs="id"
#     )
    
#     neuron_inf = {}
#     for idx, unit_id in enumerate(unit_ids_list_merged):
#         neuron_inf[unit_id] = {
#             'location_x': float(unit_locations_merged[idx, 0]),
#             'location_y': float(unit_locations_merged[idx, 1]),
#             'position_waveform': position_waveforms_merged[idx],
#             'extremum_channel': extremum_channels_merged[unit_id]
#         }
    
#     sorting_final = analyzer_merged.sorting
#     extremum_channels_final = extremum_channels_merged
# else:
#     position_waveforms = []
#     for unit_id in unit_ids_list:
#         unit_index = analyzer_curated_phy.sorting.id_to_index(unit_id)
#         template_dense_unit = templates_dense[unit_index, :, :]
#         template_sparse_unit = sparsity.sparsify_waveforms(template_dense_unit[np.newaxis, :, :], unit_id)[0]
#         sparse_channel_indices = sparsity.unit_id_to_channel_indices[unit_id]
        
#         if len(sparse_channel_indices) == 0:
#             position_waveform = np.zeros(templates_dense.shape[1], dtype=templates_dense.dtype)
#             position_waveforms.append(position_waveform)
#             continue
        
#         sparse_channel_locations = channel_locations[sparse_channel_indices, :2]
#         unit_location = unit_locations[unit_index, :2]
        
#         distances = np.sqrt(np.sum((sparse_channel_locations - unit_location[np.newaxis, :])**2, axis=1))
#         epsilon = 1e-10
#         weights = 1.0 / (distances + epsilon)
#         weights = weights / np.sum(weights)
        
#         position_waveform = np.dot(template_sparse_unit, weights)
#         position_waveforms.append(position_waveform)
    
#     position_waveforms = np.array(position_waveforms)
    
#     extremum_channels = get_template_extremum_channel(
#         analyzer_curated_phy, 
#         peak_sign="neg",
#         outputs="id"
#     )
    
#     neuron_inf = {}
#     for idx, unit_id in enumerate(unit_ids_list):
#         neuron_inf[unit_id] = {
#             'location_x': float(unit_locations[idx, 0]),
#             'location_y': float(unit_locations[idx, 1]),
#             'position_waveform': position_waveforms[idx],
#             'extremum_channel': extremum_channels[unit_id]
#         }
    
#     sorting_final = analyzer_curated_phy.sorting
#     extremum_channels_final = extremum_channels

# spike_vector = sorting_final.to_spike_vector()

# gt_detect_data = []
# for spike in spike_vector:
#     unit_index = spike['unit_index']
#     unit_id = sorting_final.unit_ids[unit_index]
#     sample_index = spike['sample_index']
#     segment_index = spike['segment_index']
    
#     time_seconds = sorting_final.sample_index_to_time(sample_index, segment_index=segment_index)
#     extremum_channel = extremum_channels_final[unit_id]
    
#     gt_detect_data.append({
#         'time': time_seconds,
#         'unit_id': unit_id,
#         'extremum_channel': str(extremum_channel)
#     })

# gt_detect_array = pd.DataFrame(gt_detect_data)

# with open(output_folder + '/neuron_inf.pickle', 'wb') as f:
#     pickle.dump(neuron_inf, f)

# gt_detect_array.to_csv(output_folder + '/gt_detect_array.csv', index=False)

# print(f"neuron_inf已保存到: {output_folder + '/neuron_inf.pickle'}")
# print(f"包含 {len(neuron_inf)} 个neurons")
# print(f"gt_detect_array已保存到: {output_folder + '/gt_detect_array.csv'}")
# print(f"包含 {len(gt_detect_array)} 个spikes")
# print(f"gt_detect_array列: {gt_detect_array.columns.tolist()}")

In [ ]:
# Clique级别训练流程
# 加载数据
with open(output_dir + '/neuron_inf.pickle', 'rb') as f:
    neuron_inf_dict = pickle.load(f)
gt_detect_array = pd.read_csv(output_dir + '/gt_detect_array.csv')

# 转换为DataFrame
neuron_inf = neuron_inf_dict_to_dataframe(neuron_inf_dict)

# 构建cliques
probe = read_probeinterface('/media/ubuntu/sda/mouse_test/probe/tip_probe_128_1.json')
cliques = build_shank_cliques(probe, shank_boundaries=[250, 750, 1250])

# 对每个clique进行训练
#for clique in cliques[0]:
clique = cliques[0]
print(f"\n{'='*60}")
print(f"Processing Clique {clique.clique_id}")
print(f"{'='*60}")

# 1. 获取recording_clique
recording_clique = get_recording_clique(recording_f, clique)
print(f"Recording clique channels: {len(recording_clique.get_channel_ids())}")

# 2. 筛选neuron_inf_clique
neuron_inf_clique = filter_neuron_inf_by_clique(neuron_inf, recording_clique)
print(f"Neurons in clique: {len(neuron_inf_clique)}")

# 3. 筛选gt_detect_array_clique
gt_detect_array_clique = filter_gt_detect_array_by_clique(gt_detect_array, recording_clique)
print(f"Spikes in clique: {len(gt_detect_array_clique)}")


clique_save_dir = output_folder + f'/clique_{clique.clique_id}'
train_data_dir = prepare_training_data(
    recording_f=recording_clique,
    gt_detect_array=gt_detect_array_clique,
    neuron_inf=neuron_inf_clique,
    save_dir=clique_save_dir,
    duration_seconds=200,
    thr_min=3,
    thr_max=10,
    distance=3,
    wlen=5,
    prominence=15,
    left_sample=10,
    right_sample=20
)

# # 6. 训练模型
model_save_dir = clique_save_dir + '/model'
n_channels = recording_clique.get_num_channels()
autosort_model, training_log = train_autosort_model(
    train_data_dir=train_data_dir,
    model_save_dir=model_save_dir,
    n_channels=n_channels,
    left_sample=10,
    right_sample=20,
    epochs=20,
    batch_size=512,
    device=None,
    early_stopping=True,
    patience=5,
    min_delta=0.0
)

print(f"Clique {clique.clique_id} training completed!")


[INFO] Shank 0: 32 channels (x range: min to 250)
[INFO] Shank 1: 32 channels (x range: 250 to 750)
[INFO] Shank 2: 32 channels (x range: 750 to 1250)
[INFO] Shank 3: 32 channels (x range: 1250 to max)
[INFO] Built 4 cliques from 4 shanks

Processing Clique 0
Recording clique channels: 32
Neurons in clique: 26
Spikes in clique: 196025
### 1. Threshold Detection
Sampling rate: 10000.0 Hz, Number of channels: 32
Recording total length: 18627072 samples (1862.71 seconds)
Will process first 2000000 samples (200.00 seconds)
Data shape: (2000000, 32) (clique channels)
Using 23 valid channels from neuron extremum_channels
Building detect_array...
Number of detected spikes: 212397

### 2. Load Ground Truth and Match
Building gt_array from gt_detect_array...
GT spike count: 31161
---spike detection rate: 0.9545
Number of matched spikes: 29744
Number of unmatched spikes: 182653

### 3. Extract Waveforms


Extracting waveforms: 100%|██████████| 30/30 [00:04<00:00,  7.08it/s]


Waveform extraction completed!
waveform shape: (212391, 32, 30)

### 4. Save Training Data
Save directory: /media/ubuntu/sda/mouse_test/sorted/mountainsort/mouse1/1215/clique_0/train_data
  ✓ neuron_mapping.pkl saved
Saving data...
  ✓ X_waveform.pkl saved
  ✓ Y_spike_id.pkl saved
  ✓ Y_spike_id_noise.pkl saved
  ✓ X_spiketrain_time.pkl saved

All data saved to: /media/ubuntu/sda/mouse_test/sorted/mountainsort/mouse1/1215/clique_0/train_data
Data statistics:
  - Total spike count: 212391
  - Number of channels: 32
  - Window length: 30
  - Number of unique units: 26
  - Noise spike count: 182647
  - Valid spike count: 29744
Using device: cuda
Create dataset...
Dataset loaded:
  - Total samples: 212391
  - Number of channels: 32
  - Window length: 30
  - Number of unique units: 26
  - Noise samples: 182647.0
  - Non-noise samples: 29744.0
Model parameters:
  - Number of channels: 32
  - Window length: 30
  - Number of units: 26
  - Input dimension: 990
Unit ID list saved to: /media/ubun

Training: 100%|██████████| 332/332 [00:02<00:00, 126.20it/s]


epoch : 1/20, detection loss = 430.011898, classification loss = 999.155978


...

epoch : 5/20, val detection loss = 347.983281, classification loss = 269.417883
Validation Loss Decreased(637.271009--->617.401164)
epoch : 6/20


Training: 100%|██████████| 332/332 [00:02<00:00, 141.69it/s]


epoch : 6/20, detection loss = 87.431249, classification loss = 141.427608


Validation: 100%|██████████| 83/83 [00:00<00:00, 226.42it/s]

epoch : 6/20, val detection loss = 440.353979, classification loss = 254.960061

Early stopping triggered: 5 consecutive epochs without improvement
Best accuracy: 0.822477 (Epoch 1)

Training completed! Training log saved to: /media/ubuntu/sda/mouse_test/sorted/mountainsort/mouse1/1215/clique_0/model/training_log.csv
Best validation accuracy: 0.822477 (Epoch 1)
Clique 0 training completed!
